### Initialization

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql.functions import *


spark = SparkSession.builder.appName("MyApp").getOrCreate()

df = spark.read.csv("/Volumes/mycatalog/myschema/myvolume/Indian_Kids_Screen_Time.csv", header=True, inferSchema=True)

my_schema = StructType([
    StructField('Age', IntegerType()),
    StructField('Gender', StringType()),
    StructField('Avg_Daily_Screen_Time_hr', DecimalType(4,2)),
    StructField('Primary_Device', StringType()),
    StructField('Exceeded_Recommended_Limit', BooleanType()),
    StructField('Educational_to_Recreational_Ratio', DecimalType(4,2)),
    StructField('Health_Impacts', StringType()),
    StructField('Urban_or_Rural', StringType())
])

new_df = spark.read.schema(my_schema).csv("/Volumes/mycatalog/myschema/myvolume/Indian_Kids_Screen_Time.csv", header = True)

#SELECT *, row_number() over() as id
#FROM table
new_df = new_df.withColumn('id', row_number().over(Window.partitionBy(lit(1)).orderBy(lit(1))))

### SELECT


In [0]:
df1 = new_df.limit(10)

#SELECT * 
#FROM table
df1.display()

#SELECT age, gender 
#FROM table
df1.select('age', 'gender').display() 

### WHERE

In [0]:
df2 = new_df.limit(10)

#SELECT * FROM table 
#WHERE age > 15 AND gender = 'Male'

df1.where((col('age') > 15) & (col('gender') == 'Male')).display()
df1.where('age > 15 AND gender = "Male"').display()

### GROUP BY

In [0]:
df3 = new_df.limit(10)

#SELECT gender, count(*) as Total 
#FROM table 
#GROUP BY gender
# df3.groupBy('gender').count().display()
# df3.groupBy('gender').agg(count('*').alias('Total')).display()

#SELECT gender, sum(age) as sum_of_age 
#FROM table 
#GROUP BY gender
df3.groupBy('gender').agg(sum('age').alias('sum_of_age')).display()


### GROUP BY + HAVING

In [0]:
df4 = new_df.limit(10)

#SELECT gender, avg(age) as avg_age 
#FROM table 
#GROUP BY gender 
#HAVING avg(age) > 14

df4.groupBy('gender').agg(avg('age').alias('avg_age')).where(col('avg_age') > 14).display()





### ORDER BY

In [0]:
df5 = new_df.limit(10)

#SELECT * 
#FROM table 
#ORDER BY age, Avg_Daily_Screen_Time_hr DESC

df5.orderBy(col('age'), col('Avg_Daily_Screen_Time_hr').desc()).display()

### WHERE + GROUP BY + HAVING

In [0]:
df6 = new_df.limit(10)

#SELECT gender, primary_device, avg(avg_daily_screen_time_hr) as avg_screen_time
#FROM table 
#WHERE gender = 'Female' 
#GROUP BY gender,  primary_device
#having avg(Avg_Daily_Screen_Time_hr) > 3
#ORDER BY avg(avg_daily_screen_time_hr) DESC

df6.groupBy('Gender', 'primary_device').agg(avg('avg_daily_screen_time_hr').alias('avg_screen_time')).where('Gender = "Female" and avg_screen_time > 3').orderBy(col('avg_screen_time').desc()).display()


### JOIN

In [0]:
df7 = new_df.select('id', 'age').where('id < 11')
df8 = new_df.select('id', 'gender').where('id > 3 and id < 14')

#SELECT a.id, a.age, b.gender
#FROM df7 a
#JOIN/LEFT JOIN/RIGHT/JOIN/FULL JOIN df8 b
#ON a.id = b.id

df7.join(df8, df7.id == df8.id, 'inner').select(df7.id, df7.age, df8.id, df8.gender).display()
df7.join(df8, df7.id == df8.id, 'left').select(df7.id, df7.age, df8.id, df8.gender).display()
df7.join(df8, df7.id == df8.id, 'right').select(df7.id, df7.age, df8.id, df8.gender).display()
df7.join(df8, df7.id == df8.id, 'outer').select(df7.id, df7.age, df8.id, df8.gender).display()



### UNION and UNION ALL

In [0]:
df9 = new_df.select('id', 'age').where('id < 11')
df10 = new_df.select('id', 'gender').where('id > 5 and id < 16')

#SELECT id FROM table1
#UNION
#SELECT id from table2
df9.select('id').union(df10.select('id')).distinct().display()

#SELECT id FROM table1
#UNION ALL
#SELECT id from table2
df9.select('id').unionAll(df10.select('id')).display()

### ALIAS

In [0]:
df12 = new_df.limit(10)

#SELECT age as kid_age
#FROM table
df12.select(col('age').alias('kid_age')).display()


### IN

In [0]:
df13 = new_df.limit(10)

#SELECT *
#FROM table
#WHERE age in (11,12)

df13.where(col('age').isin(11,12)).display()

### BETWEEN

In [0]:
df14 = new_df.limit(10)

# SELECT * 
# FROM table 
# WHERE age BETWEEN 11 AND 15;

df14.where('age >= 11 and age <= 15').display()

### LIKE

In [0]:
df15 = new_df.limit(10)

# SELECT * 
# FROM table 
# WHERE primary_device LIKE 'S%';

df15.where(col('primary_device').like('S%')).display()

### IS NULL, IS NOT NULL

In [0]:
df16 = new_df.limit(10)

# SELECT * 
# FROM table 
# WHERE primary_device IS NULL;

df16.where(col('primary_device').isNull()).display()

# SELECT * 
# FROM table 
# WHERE primary_device IS NOT NULL;

df16.where(col('primary_device').isNotNull()).display()